# Phase 7 — Error Analysis & Final Report

**All Members** | Steps 7.1 – 7.3

| Step | Description |
|---|---|
| 7.1 | Identify false positive and false negative cohorts; characterize by user frequency and product popularity |
| 7.2 | Analyze cold-start performance (users with < 5 orders) across all models |
| 7.3 | Compile final comparison table with all metrics across all four models |

**Prerequisite:** Run all previous notebooks (01 – 05) first.

In [ ]:
import os
import sys
import json
import joblib
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

sys.path.insert(0, '..')
from utils.metrics import print_metrics_table

FEATURES_DIR = '../features'
MODELS_DIR   = '../saved_models'
OUTPUTS_DIR  = '../outputs'

sns.set_style('darkgrid')
print('Setup complete.')

In [ ]:
feature_df = pd.read_parquet(f'{FEATURES_DIR}/feature_matrix.parquet')

FEATURE_COLS = [
    'total_orders', 'avg_basket_size', 'avg_days_between_orders', 'user_reorder_rate',
    'product_order_freq', 'product_reorder_rate', 'avg_add_to_cart_position',
    'up_times_ordered', 'up_times_reordered', 'up_reorder_ratio',
    'up_days_since_last', 'up_order_streak',
    'order_dow', 'order_hour_of_day', 'days_since_last_order', 'purchase_velocity',
]

test_df = feature_df[feature_df['split'] == 'test'].copy()
X_test  = test_df[FEATURE_COLS]
y_test  = test_df['reordered']

print(f'Test set: {len(test_df):,} rows, {test_df["user_id"].nunique():,} users')

In [ ]:
# Load GBDT model (best tabular model) for error analysis
gbdt_model = lgb.Booster(model_file=f'{MODELS_DIR}/lightgbm_model.txt')
gbdt_proba = gbdt_model.predict(X_test)

test_df = test_df.copy()
test_df['pred_proba'] = gbdt_proba
test_df['pred_label'] = (gbdt_proba >= 0.5).astype(int)

print('GBDT predictions loaded.')
print(f'Predicted positives: {test_df["pred_label"].sum():,}')
print(f'Actual positives:    {y_test.sum():,}')

## Step 7.1 — False Positive & False Negative Analysis

In [ ]:
tp_mask = (test_df['pred_label'] == 1) & (test_df['reordered'] == 1)
fp_mask = (test_df['pred_label'] == 1) & (test_df['reordered'] == 0)
fn_mask = (test_df['pred_label'] == 0) & (test_df['reordered'] == 1)
tn_mask = (test_df['pred_label'] == 0) & (test_df['reordered'] == 0)

tp_df = test_df[tp_mask]
fp_df = test_df[fp_mask]
fn_df = test_df[fn_mask]

print('=== Confusion Matrix Breakdown ===')
print(f'True Positives  (TP): {tp_mask.sum():>8,}')
print(f'False Positives (FP): {fp_mask.sum():>8,}  <- predicted reorder, actually not')
print(f'False Negatives (FN): {fn_mask.sum():>8,}  <- missed reorder (highest cost)')
print(f'True Negatives  (TN): {tn_mask.sum():>8,}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Total orders distribution (TP vs FP)
axes[0, 0].hist(tp_df['total_orders'], bins=30, color='#22c55e', alpha=0.7, label='True Positives', density=True)
axes[0, 0].hist(fp_df['total_orders'], bins=30, color='#ef4444', alpha=0.7, label='False Positives', density=True)
axes[0, 0].set_title('User Order History: TP vs FP', fontsize=12)
axes[0, 0].set_xlabel('Total Prior Orders')
axes[0, 0].set_ylabel('Density')
axes[0, 0].legend()

# Plot 2: Reorder ratio (TP vs FN)
axes[0, 1].hist(tp_df['up_reorder_ratio'], bins=30, color='#22c55e', alpha=0.7, label='True Positives', density=True)
axes[0, 1].hist(fn_df['up_reorder_ratio'], bins=30, color='#f97316', alpha=0.7, label='False Negatives', density=True)
axes[0, 1].set_title('User-Product Reorder Ratio: TP vs FN', fontsize=12)
axes[0, 1].set_xlabel('Reorder Ratio')
axes[0, 1].set_ylabel('Density')
axes[0, 1].legend()

# Plot 3: Product popularity (TP vs FP)
axes[1, 0].hist(np.log1p(tp_df['product_order_freq']), bins=30, color='#22c55e', alpha=0.7,
                label='True Positives', density=True)
axes[1, 0].hist(np.log1p(fp_df['product_order_freq']), bins=30, color='#ef4444', alpha=0.7,
                label='False Positives', density=True)
axes[1, 0].set_title('Product Popularity (log): TP vs FP', fontsize=12)
axes[1, 0].set_xlabel('log(Product Order Frequency)')
axes[1, 0].set_ylabel('Density')
axes[1, 0].legend()

# Plot 4: Days since last ordered (TP vs FN)
axes[1, 1].hist(tp_df['up_days_since_last'].clip(upper=30), bins=30, color='#22c55e', alpha=0.7,
                label='True Positives', density=True)
axes[1, 1].hist(fn_df['up_days_since_last'].clip(upper=30), bins=30, color='#f97316', alpha=0.7,
                label='False Negatives', density=True)
axes[1, 1].set_title('Days Since Last Order: TP vs FN', fontsize=12)
axes[1, 1].set_xlabel('Days Since Last Ordered (capped at 30)')
axes[1, 1].set_ylabel('Density')
axes[1, 1].legend()

plt.suptitle('GBDT Error Analysis — False Positive & False Negative Cohorts', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUTS_DIR}/07_error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nKey observations:')
print('  FP cohort: more common for low-popularity products and users with moderate history')
print('  FN cohort: items with low reorder ratio or long time since last purchase')

## Step 7.2 — Cold-Start Analysis (users with < 5 prior orders)

In [ ]:
cold_df = test_df[test_df['total_orders'] < 5]
warm_df = test_df[test_df['total_orders'] >= 5]

def cohort_metrics(df, model_name=''):
    if len(df) == 0 or df['reordered'].sum() == 0:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'n': 0, 'n_users': 0}
    return {
        'precision': precision_score(df['reordered'], df['pred_label'], zero_division=0),
        'recall':    recall_score(df['reordered'], df['pred_label'], zero_division=0),
        'f1':        f1_score(df['reordered'], df['pred_label'], zero_division=0),
        'n':         len(df),
        'n_users':   df['user_id'].nunique(),
    }

cold_m = cohort_metrics(cold_df)
warm_m = cohort_metrics(warm_df)

print('=== Cold-Start vs Warm-Start Performance (GBDT) ===')
print(f'Cold-start users (<5 orders):')
print(f'  Users:     {cold_m["n_users"]:,}')
print(f'  Precision: {cold_m["precision"]:.4f}')
print(f'  Recall:    {cold_m["recall"]:.4f}')
print(f'  F1:        {cold_m["f1"]:.4f}')
print()
print(f'Warm-start users (>=5 orders):')
print(f'  Users:     {warm_m["n_users"]:,}')
print(f'  Precision: {warm_m["precision"]:.4f}')
print(f'  Recall:    {warm_m["recall"]:.4f}')
print(f'  F1:        {warm_m["f1"]:.4f}')

In [ ]:
# Performance by order count bucket
buckets  = [1, 2, 3, 5, 10, 20, 50, 100, 9999]
labels   = ['1', '2', '3', '4', '5-9', '10-19', '20-49', '50-99', '100+']

bucket_results = []
for i in range(len(buckets) - 1):
    lo, hi = buckets[i], buckets[i + 1]
    sub = test_df[(test_df['total_orders'] >= lo) & (test_df['total_orders'] < hi)]
    if len(sub) == 0 or sub['reordered'].sum() == 0:
        continue
    bucket_results.append({
        'bucket': labels[i],
        'n_users': sub['user_id'].nunique(),
        'f1':      f1_score(sub['reordered'], sub['pred_label'], zero_division=0),
        'recall':  recall_score(sub['reordered'], sub['pred_label'], zero_division=0),
    })

bucket_df = pd.DataFrame(bucket_results)

fig, ax = plt.subplots(figsize=(12, 5))
x     = np.arange(len(bucket_df))
width = 0.35
ax.bar(x - width / 2, bucket_df['f1'],    width, label='F1',    color='#6366f1')
ax.bar(x + width / 2, bucket_df['recall'], width, label='Recall', color='#22c55e')
ax.set_xticks(x)
ax.set_xticklabels(bucket_df['bucket'])
ax.set_xlabel('Prior Orders per User', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('GBDT Performance by User Order History Depth (Cold-Start Analysis)', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(0, 1)

# Annotate with user count
for i, row in bucket_df.iterrows():
    ax.text(i, -0.08, f'n={row["n_users"]}', ha='center', va='top', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig(f'{OUTPUTS_DIR}/07_cold_start.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7.3 — Final Four-Model Comparison Table

In [ ]:
with open(f'{OUTPUTS_DIR}/model_results.json', 'r') as f:
    results = json.load(f)

print('=' * 72)
print('  CartIQ ML — Final Model Comparison (Test Set)')
print('=' * 72)
print_metrics_table(results)
print('Notes:')
print('  RF / GBDT / NCF : binary classification at threshold=0.5')
print('  GRU             : top-K recommendation (Precision@10, Recall@10)')

In [ ]:
# Summary insights
best_f1_model = max(results, key=lambda m: results[m].get('f1', 0))
best_f1       = results[best_f1_model]['f1']

rf_f1   = results.get('random_forest', {}).get('f1', 0)
gbdt_f1 = results.get('gbdt', {}).get('f1', 0)
ncf_f1  = results.get('ncf', {}).get('f1', 0)

print('=== Key Insights ===')
print(f'1. Best overall model (F1): {best_f1_model.upper()} ({best_f1:.4f})')
print(f'2. GBDT improvement over RF baseline: {gbdt_f1 - rf_f1:+.4f} F1 points')
print(f'3. NCF vs RF (embedding-based vs tabular): {ncf_f1 - rf_f1:+.4f} F1 points')
print()
print('Cold-start finding:')
print(f'  Cold users (<5 orders) F1:  {cold_m["f1"]:.4f}')
print(f'  Warm users (>=5 orders) F1: {warm_m["f1"]:.4f}')
print(f'  Delta: {warm_m["f1"] - cold_m["f1"]:+.4f}  (cold-start gap)')
print()
print('Recommendation: GBDT is the production model for the CartIQ app.')
print('  NCF embeddings add value for discovery recommendations.')
print('  GRU captures sequential patterns for next-basket prediction.')

In [ ]:
# Save final results summary
summary = {
    'model_results': results,
    'cold_start': {
        'cold': cold_m,
        'warm': warm_m,
    },
    'best_model': best_f1_model,
    'best_f1': best_f1,
}

with open(f'{OUTPUTS_DIR}/final_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Final summary saved to {OUTPUTS_DIR}/final_summary.json')
print('Phase 7 complete. CartIQ ML pipeline finished.')